# Air Quality Time-Series Modeling
Build a time-series regression model on prepared air-quality data and export evaluation metrics for the frontend.

**Output:** `frontend/public/air_quality_model_metrics.json`

In [24]:
import json
from datetime import datetime
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
 )
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")

HORIZON_HOURS = 3
LAGS = [1, 2, 3, 6, 12, 24]
TEST_FRACTION = 0.2
RANDOM_STATE = 42

def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "frontend").exists() and (cwd / "backend").exists():
        return cwd
    for parent in [cwd, *cwd.parents]:
        if (parent / "frontend").exists() and (parent / "backend").exists():
            return parent
    return cwd

REPO_ROOT = resolve_repo_root()
DATA_PATH = REPO_ROOT / "backend" / "services" / "air_quality" / "prepared_risk_data.csv"

DATA_PATH

WindowsPath('C:/Users/Taha/Desktop/h12_gh/h12_wasted_potential/backend/services/air_quality/prepared_risk_data.csv')

In [25]:
df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").set_index("timestamp")

df.head()

,aqi,so2,pm10,wind_speed,wind_direction,cardinal,dist_gabes_city,dist_mareth,dist_matmata,dist_teboulbou
timestamp,,,,,,,,,,
2026-04-11 19:00:00,4,0.04,109.44,10.1,109,E,5.62,44.45,47.66,9.62
2026-04-11 20:00:00,4,0.04,109.21,10.0,107,E,5.62,44.45,47.66,9.62
2026-04-11 21:00:00,4,0.05,104.72,12.1,94,E,5.62,44.45,47.66,9.62
2026-04-11 22:00:00,3,0.06,98.10,13.9,104,E,5.62,44.45,47.66,9.62
2026-04-11 23:00:00,4,0.21,103.38,12.6,109,E,5.62,44.45,47.66,9.62


In [26]:
TARGET_COLS = ["pm10", "so2"]

missing = [col for col in TARGET_COLS if col not in df.columns]
if missing:
    raise ValueError(f"Missing target columns: {missing}")

# Exogenous time and weather signals
exog_df = pd.DataFrame(index=df.index)
exog_df["hour"] = df.index.hour
exog_df["dayofweek"] = df.index.dayofweek
exog_df["hour_sin"] = np.sin(2 * np.pi * exog_df["hour"] / 24)
exog_df["hour_cos"] = np.cos(2 * np.pi * exog_df["hour"] / 24)
exog_df["dow_sin"] = np.sin(2 * np.pi * exog_df["dayofweek"] / 7)
exog_df["dow_cos"] = np.cos(2 * np.pi * exog_df["dayofweek"] / 7)

for col in ["aqi", "wind_speed", "wind_direction"]:
    if col in df.columns:
        exog_df[col] = df[col]

# Horizon-shifted targets (index t predicts t + H)
target_df = df[TARGET_COLS].shift(-HORIZON_HOURS)
target_df.columns = [f"{col}_t_plus_{HORIZON_HOURS}h" for col in TARGET_COLS]

dataset = pd.concat([target_df, exog_df], axis=1).dropna()
y_all = dataset[[f"{col}_t_plus_{HORIZON_HOURS}h" for col in TARGET_COLS]]
y_all.columns = TARGET_COLS
exog_all = dataset.drop(columns=[f"{col}_t_plus_{HORIZON_HOURS}h" for col in TARGET_COLS])

split_idx = int(len(dataset) * (1 - TEST_FRACTION))
y_train = y_all.iloc[:split_idx]
y_test = y_all.iloc[split_idx:]
exog_train = exog_all.iloc[:split_idx] if exog_all.shape[1] else None
exog_test = exog_all.iloc[split_idx:] if exog_all.shape[1] else None

y_train.shape, y_test.shape, exog_all.shape

((132, 2), (33, 2), (165, 9))

In [27]:
SEASONAL_PERIOD = 24
ORDER_CANDIDATES = [(1, 0, 1), (0, 1, 1), (1, 1, 1)]
SEASONAL_CANDIDATES = [(0, 0, 0, 0), (1, 0, 1, SEASONAL_PERIOD)]
VAL_SIZE = max(12, HORIZON_HOURS * 4)

def fit_sarimax(series, exog, order, seasonal_order):
    model = SARIMAX(
        series,
        exog=exog,
        order=order,
        seasonal_order=seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    return model.fit(disp=False, maxiter=80)

def fit_ets(series, trend, seasonal, damped):
    model = ExponentialSmoothing(
        series,
        trend=trend,
        seasonal=seasonal,
        seasonal_periods=SEASONAL_PERIOD if seasonal else None,
        damped_trend=damped,
        initialization_method="estimated",
    )
    return model.fit(optimized=True)

def select_best_model(series, exog):
    seasonal_candidates = SEASONAL_CANDIDATES
    ets_candidates = [
        {"trend": None, "seasonal": None, "damped": False},
        {"trend": "add", "seasonal": None, "damped": True},
        {"trend": "add", "seasonal": "add", "damped": True},
    ]
    if len(series) < SEASONAL_PERIOD * 2:
        seasonal_candidates = [(0, 0, 0, 0)]
        ets_candidates = [
            {"trend": None, "seasonal": None, "damped": False},
            {"trend": "add", "seasonal": None, "damped": True},
        ]

    if len(series) <= VAL_SIZE + 5:
        return {"type": "sarimax", "order": ORDER_CANDIDATES[0], "seasonal_order": seasonal_candidates[0]}

    train = series.iloc[:-VAL_SIZE]
    val = series.iloc[-VAL_SIZE:]
    exog_train = exog.iloc[:-VAL_SIZE] if exog is not None else None
    exog_val = exog.iloc[-VAL_SIZE:] if exog is not None else None

    best = None
    best_rmse = float("inf")

    for order in ORDER_CANDIDATES:
        for seasonal_order in seasonal_candidates:
            try:
                result = fit_sarimax(train, exog_train, order, seasonal_order)
                forecast = result.get_forecast(steps=len(val), exog=exog_val).predicted_mean
                rmse = mean_squared_error(val, forecast, squared=False)
                if rmse < best_rmse:
                    best = {"type": "sarimax", "order": order, "seasonal_order": seasonal_order}
                    best_rmse = rmse
            except Exception:
                continue

    for cfg in ets_candidates:
        try:
            result = fit_ets(train, cfg["trend"], cfg["seasonal"], cfg["damped"])
            forecast = result.forecast(len(val))
            rmse = mean_squared_error(val, forecast, squared=False)
            if rmse < best_rmse:
                best = {"type": "ets", **cfg}
                best_rmse = rmse
        except Exception:
            continue

    return best if best else {"type": "sarimax", "order": ORDER_CANDIDATES[0], "seasonal_order": seasonal_candidates[0]}

models = {}
model_specs = {}
predictions = {}
for col in TARGET_COLS:
    spec = select_best_model(y_train[col], exog_train)
    if spec["type"] == "sarimax":
        result = fit_sarimax(y_train[col], exog_train, spec["order"], spec["seasonal_order"])
        forecast = result.get_forecast(steps=len(y_test), exog=exog_test).predicted_mean
    else:
        result = fit_ets(y_train[col], spec["trend"], spec["seasonal"], spec["damped"])
        forecast = result.forecast(len(y_test))

    models[col] = result
    model_specs[col] = spec
    predictions[col] = pd.Series(forecast.values, index=y_test.index)

y_pred = pd.DataFrame(predictions)

y_pred.head()

,pm10,so2
timestamp,,
2026-04-17 07:00:00,45.171645,0.184795
2026-04-17 08:00:00,46.983081,0.200815
2026-04-17 09:00:00,48.638539,0.215444
2026-04-17 10:00:00,50.151450,0.231249
2026-04-17 11:00:00,51.534088,0.258729


In [28]:
def regression_metrics(y_true, y_pred):
    return {
        "r2": float(r2_score(y_true, y_pred)),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(mean_squared_error(y_true, y_pred, squared=False)),
        "mape": float(mean_absolute_percentage_error(y_true, y_pred)),
    }

def regression_metrics_safe(y_true, y_pred):
    y_true = pd.Series(y_true)
    y_pred = pd.Series(y_pred)
    mask = ~(y_true.isna() | y_pred.isna())
    if mask.sum() == 0:
        return {"r2": None, "mae": None, "rmse": None, "mape": None}
    return regression_metrics(y_true[mask], y_pred[mask])

baseline_pred = pd.DataFrame(index=y_test.index)
seasonal_pred = pd.DataFrame(index=y_test.index)
seasonal_shift = 24 - HORIZON_HOURS
for col in TARGET_COLS:
    baseline_pred[col] = df.loc[y_test.index, col]
    seasonal_pred[col] = df[col].shift(seasonal_shift).loc[y_test.index]

metrics = {}
for col in TARGET_COLS:
    metrics[col] = regression_metrics(y_test[col], y_pred[col])
    metrics[col]["baseline"] = regression_metrics(y_test[col], baseline_pred[col])
    metrics[col]["baseline_seasonal"] = regression_metrics_safe(y_test[col], seasonal_pred[col])

metrics_table = pd.DataFrame(
    {
        col: {
            "r2": metrics[col]["r2"],
            "rmse": metrics[col]["rmse"],
            "mae": metrics[col]["mae"],
            "mape": metrics[col]["mape"],
            "baseline_rmse": metrics[col]["baseline"]["rmse"],
            "baseline_mae": metrics[col]["baseline"]["mae"],
            "baseline_mape": metrics[col]["baseline"]["mape"],
            "seasonal_rmse": metrics[col]["baseline_seasonal"]["rmse"],
            "seasonal_mae": metrics[col]["baseline_seasonal"]["mae"],
            "seasonal_mape": metrics[col]["baseline_seasonal"]["mape"],
        }
        for col in TARGET_COLS
    }
).T

metrics_table

,r2,rmse,mae,mape,baseline_rmse,baseline_mae,baseline_mape,seasonal_rmse,seasonal_mae,seasonal_mape
pm10,-19.669706,30.098179,29.326241,1.069248,7.343896,5.488788,0.207692,6.770844,4.972727,0.161103
so2,-19.540508,0.326755,0.281716,2.442738,0.090470,0.070000,0.514149,0.636251,0.399697,2.775270


## Benchmark summary
This benchmark compares the model against persistence (last observed value) and a seasonal naive baseline (same hour previous day).
Interpretation:
- `good` = model RMSE improves the chosen baseline by at least 10%.
- `parity` = model RMSE within 10% of the chosen baseline.
- `worse` = model RMSE more than 10% worse than the chosen baseline.

In [29]:
def benchmark_label(model_rmse, baseline_rmse, tolerance=0.10):
    if baseline_rmse == 0:
        return "unknown"
    ratio = model_rmse / baseline_rmse
    if ratio <= 1 - tolerance:
        return "good"
    if ratio <= 1 + tolerance:
        return "parity"
    return "worse"

def choose_baseline(entry):
    persistence_rmse = entry["baseline"]["rmse"]
    seasonal_rmse = entry.get("baseline_seasonal", {}).get("rmse")
    if seasonal_rmse is not None and seasonal_rmse < persistence_rmse:
        return "seasonal", seasonal_rmse
    return "persistence", persistence_rmse

benchmark_rows = []
for col in TARGET_COLS:
    model_rmse = metrics[col]["rmse"]
    baseline_type, baseline_rmse = choose_baseline(metrics[col])
    benchmark_rows.append({
        "target": col,
        "model_rmse": model_rmse,
        "baseline_rmse": baseline_rmse,
        "baseline_type": baseline_type,
        "rmse_ratio": model_rmse / baseline_rmse if baseline_rmse else None,
        "benchmark": benchmark_label(model_rmse, baseline_rmse),
        "r2": metrics[col]["r2"],
    })

benchmark_table = pd.DataFrame(benchmark_rows)
benchmark_table

,target,model_rmse,baseline_rmse,baseline_type,rmse_ratio,benchmark,r2
0,pm10,30.098179,6.770844,seasonal,4.445263,worse,-19.669706
1,so2,0.326755,0.090470,persistence,3.611749,worse,-19.540508


In [30]:
output = {
    "meta": {
        "generated_at": datetime.utcnow().isoformat() + "Z",
        "horizon_hours": HORIZON_HOURS,
        "lags": LAGS,
        "train_rows": int(len(y_train)),
        "test_rows": int(len(y_test)),
        "dataset_start": df.index.min().isoformat(),
        "dataset_end": df.index.max().isoformat(),
        "model": "best-of SARIMAX/ETS",
        "model_specs": model_specs,
    },
    "metrics": metrics,
    "benchmark": benchmark_table.to_dict(orient="records"),
}

frontend_public = REPO_ROOT / "frontend" / "public"
frontend_public.mkdir(parents=True, exist_ok=True)
output_path = frontend_public / "air_quality_model_metrics.json"
output_path.write_text(json.dumps(output, indent=2))

output_path

WindowsPath('C:/Users/Taha/Desktop/h12_gh/h12_wasted_potential/frontend/public/air_quality_model_metrics.json')